In [1]:
import os
import re
import json
from tqdm import tqdm

def clean_question(text):
    if not text:
        return ""
    text = str(text)
    # 1. Loại bỏ các tiền tố như Question:, Q:, Hỏi:
    text = re.sub(r'^(?:Question|Q|Hỏi)\s*:\s*', '', text, flags=re.IGNORECASE)
    # 2. Loại bỏ các thẻ HTML/Markdown lộn xộn (<...>, ###, **, bullet *)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\*\*[^\*]+\*\*\s*', ' ', text)
    text = re.sub(r'[\*\#•\-]+\s*', ' ', text)
    # 3. Loại bỏ ký tự lạ không thuộc tiếng Việt chuẩn hoặc dấu câu cơ bản
    text = re.sub(r'[^\w\s\,\.\?\!\;\:\-\/\%\(\)]+', ' ', text)
    # 4. Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_randomqa_colab(input_path, output_jsonl_path, output_txt_path):
    if not os.path.exists(input_path):
        print(f"[ERROR] File not found: {input_path}")
        return

    print(f"[INFO] Reading raw QA file from: {input_path}")
    with open(input_path, "r", encoding="utf-8") as f:
        raw_lines = [line.strip() for line in f if line.strip()]

    print(f"[INFO] Cleaning questions from {len(raw_lines):,} records...")
    cleaned_records = []

    for line in tqdm(raw_lines, desc="Cleaning questions"):
        q_text = ""

        if line.startswith("{") and line.endswith("}"):
            try:
                data = json.loads(line)
                if "question" in data and data["question"]:
                    q_text = clean_question(data["question"])
            except Exception:
                # Fallback regex nếu JSON có dấu ngoặc kép chưa escape
                q_match = re.search(r'"question"\s*:\s*"(.*?)"\s*(?:,|\}$)', line, re.DOTALL)
                if q_match:
                    q_text = clean_question(q_match.group(1))
        else:
            q_text = clean_question(line)

        # Chỉ lấy những câu hỏi đủ độ dài và ý nghĩa (>= 10 ký tự)
        if q_text and len(q_text) >= 10:
            cleaned_records.append({"question": q_text})

    print(f"[INFO] Writing {len(cleaned_records):,} clean questions to JSONL: {output_jsonl_path}")
    with open(output_jsonl_path, "w", encoding="utf-8") as f:
        for rec in cleaned_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"[INFO] Writing {len(cleaned_records):,} clean questions to TXT: {output_txt_path}")
    with open(output_txt_path, "w", encoding="utf-8") as f:
        for rec in cleaned_records:
            f.write(rec["question"] + "\n")

    print("[SUCCESS] Completed cleaning questions on Colab!")
    print(f"[INFO] Total clean questions saved: {len(cleaned_records):,}")

# Chạy làm sạch trên Google Colab
clean_randomqa_colab(
    input_path="/content/randomqa.txt",
    output_jsonl_path="/content/cleaned_questions.jsonl",
    output_txt_path="/content/cleaned_questions.txt"
)

[INFO] Reading raw QA file from: /content/randomqa.txt
[INFO] Cleaning questions from 67,372 records...


Cleaning questions: 100%|██████████| 67372/67372 [00:04<00:00, 16693.04it/s]


[INFO] Writing 67,365 clean questions to JSONL: /content/cleaned_questions.jsonl
[INFO] Writing 67,365 clean questions to TXT: /content/cleaned_questions.txt
[SUCCESS] Completed cleaning questions on Colab!
[INFO] Total clean questions saved: 67,365
